In [1]:
import pandas as pd
import numpy as np

RAW_PATH  = "/Users/kamaleshwari/Desktop/decoding customer value/Dataset.csv"
OUT_PATH  = "/Users/kamaleshwari/Desktop/decoding customer value/customers_enriched.csv"

df = pd.read_csv(RAW_PATH)
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

Loaded: 3900 rows × 18 columns


In [2]:
df.columns = [
    "customer_id", "age", "gender", "item_purchased", "category",
    "purchase_amount", "location", "size", "color", "season",
    "review_rating", "subscription_status", "shipping_type",
    "discount_applied", "promo_code_used", "previous_purchases",
    "payment_method", "frequency_of_purchases"
]

missing_ratings = df["review_rating"].isna().sum()
median_rating   = df["review_rating"].median()
df["review_rating"] = df["review_rating"].fillna(median_rating)
print(f"Imputed {missing_ratings} missing review_rating values with median={median_rating}")

Imputed 37 missing review_rating values with median=3.8


In [3]:
assert (df["discount_applied"] == df["promo_code_used"]).all(), \
    "Assumption violated: discount_applied ≠ promo_code_used in some rows"
 

In [4]:
bool_cols = ["subscription_status", "discount_applied", "promo_code_used"]
for col in bool_cols:
    df[col + "_flag"] = (df[col] == "Yes").astype(int)

In [5]:
remaining_nulls = df.isnull().sum().sum()
assert remaining_nulls == 0, f"Still {remaining_nulls} nulls after cleaning"
print(f"Cleaning complete. Zero nulls remain.")

Cleaning complete. Zero nulls remain.


In [6]:
#feature engineering 
#frequency score
freq_map = {
    "Weekly":        7,
    "Bi-Weekly":     6,
    "Fortnightly":   5,
    "Monthly":       4,
    "Every 3 Months":3,
    "Quarterly":     2,
    "Annually":      1
}
df["frequency_score"] = df["frequency_of_purchases"].map(freq_map)

In [7]:
#promo-dependency 
df["promo_dependency_score"] = df["discount_applied_flag"]
#satisfaction flag
df["satisfaction_flag"] = (df["review_rating"] >= 4.0).astype(int)

In [8]:
#customer loyalty value
df["clv_proxy"] = (
    df["purchase_amount"]
    * df["frequency_score"]
    * np.log1p(df["previous_purchases"])
)

In [9]:
#value tier
df["value_tier"] = pd.qcut(
    df["clv_proxy"],
    q=4,
    labels=["Low", "Mid", "High", "Premium"]
)
#age group 
df["age_group"] = pd.cut(
    df["age"],
    bins=[17, 24, 34, 44, 54, 70],
    labels=["18-24", "25-34", "35-44", "45-54", "55-70"]
)

In [10]:
#recency, frequency, monetary
df["r_score"] = pd.qcut(df["frequency_score"].rank(method="first"),  5, labels=[1,2,3,4,5]).astype(int)
df["f_score"] = pd.qcut(df["previous_purchases"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
df["m_score"] = pd.qcut(df["purchase_amount"].rank(method="first"),   5, labels=[1,2,3,4,5]).astype(int)
df["rfm_total"] = df["r_score"] + df["f_score"] + df["m_score"]
 
rfm_labels = pd.cut(
    df["rfm_total"],
    bins=[2, 6, 9, 12, 15],
    labels=["At-Risk", "Needs Attention", "Loyal", "Champions"]
)
df["rfm_segment"] = rfm_labels

In [11]:
#loyalty - definition 1
median_pp = df["previous_purchases"].median()  # = 25
df["loyalty_def1"] = (
    (df["previous_purchases"] >= median_pp) &
    (df["subscription_status_flag"] == 1)
).astype(int)

#loyalty - definition 2
high_freq_labels = {"Weekly", "Bi-Weekly", "Fortnightly"}
df["loyalty_def2"] = (
    df["frequency_of_purchases"].isin(high_freq_labels) &
    (df["review_rating"] >= 4.0) &
    (df["promo_code_used_flag"] == 0)
).astype(int)

df["is_loyal"] = df["loyalty_def2"]

In [12]:
#grouping customers on the basis of loyalty and promo-dependency 
def assign_segment(row):
    if row["loyalty_def2"] == 1:
        return "Brand Advocate"
    elif row["subscription_status"] == "Yes":
        return "Subscription Member"
    elif row["promo_dependency_score"] == 1:
        return "Promo Hunter"
    else:
        return "Passive Buyer"

df["customer_segment"] = df.apply(assign_segment, axis=1)

In [13]:
print("\n─── VALIDATION SUMMARY ───────────────────────────────────")
print(f"Total customers : {len(df)}")
print(f"Features added  : 16 new columns")
print()
 
print("Value tier distribution:")
print(df["value_tier"].value_counts().sort_index().to_string())
print()
 
print("RFM segment distribution:")
print(df["rfm_segment"].value_counts().sort_index().to_string())
print()
 
print("Customer segment distribution:")
print(df["customer_segment"].value_counts().to_string())
print()
 
print("Loyalty definition comparison:")
loyal1 = df["loyalty_def1"]
loyal2 = df["loyalty_def2"]
print(f"  Def 1 count: {loyal1.sum()} ({loyal1.mean()*100:.1f}%)")
print(f"  Def 2 count: {loyal2.sum()} ({loyal2.mean()*100:.1f}%)")
print(f"  Mean CLV — Def 1 loyal: ${df.loc[loyal1==1,'clv_proxy'].mean():.2f}")
print(f"  Mean CLV — Def 2 loyal: ${df.loc[loyal2==1,'clv_proxy'].mean():.2f}")
print(f"  Mean CLV — overall:     ${df['clv_proxy'].mean():.2f}")
print(f"  CLV correlation — Def 1: {np.corrcoef(loyal1, df['clv_proxy'])[0,1]:.4f}")
print(f"  CLV correlation — Def 2: {np.corrcoef(loyal2, df['clv_proxy'])[0,1]:.4f}")
print()
 
print("Promo dependency by customer segment:")
print(df.groupby("customer_segment")["promo_dependency_score"].mean().round(2).to_string())
print()
 
print("Mean CLV proxy by value tier:")
print(df.groupby("value_tier", observed=True)["clv_proxy"].mean().round(2).to_string())
print()
 
print("Satisfaction flag by segment:")
print(df.groupby("customer_segment")["satisfaction_flag"].mean().round(3).to_string())
 
print("\n─── FINAL COLUMN LIST ────────────────────────────────────")
for col in df.columns:
    print(f"  {col}")
 


─── VALIDATION SUMMARY ───────────────────────────────────
Total customers : 3900
Features added  : 16 new columns

Value tier distribution:
value_tier
Low        976
Mid        974
High       976
Premium    974

RFM segment distribution:
rfm_segment
At-Risk             606
Needs Attention    1659
Loyal              1328
Champions           307

Customer segment distribution:
customer_segment
Passive Buyer          1830
Subscription Member    1053
Promo Hunter            624
Brand Advocate          393

Loyalty definition comparison:
  Def 1 count: 569 (14.6%)
  Def 2 count: 393 (10.1%)
  Mean CLV — Def 1 loyal: $873.57
  Mean CLV — Def 2 loyal: $1111.15
  Mean CLV — overall:     $716.38
  CLV correlation — Def 1: 0.1225
  CLV correlation — Def 2: 0.2493

Promo dependency by customer segment:
customer_segment
Brand Advocate         0.0
Passive Buyer          0.0
Promo Hunter           1.0
Subscription Member    1.0

Mean CLV proxy by value tier:
value_tier
Low         181.96
Mid      

In [14]:
df.to_csv(OUT_PATH, index=False)
print(f"\nExported enriched dataset → {OUT_PATH}")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
 


Exported enriched dataset → /Users/kamaleshwari/Desktop/decoding customer value/customers_enriched.csv
Shape: 3900 rows × 36 columns
